In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from torch.optim import AdamW
import json
from tqdm import tqdm

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
import json
import random
from tqdm import tqdm

# Config
MODEL_NAME = "bert-base-uncased"
MAX_LEN_QUERY = 32
MAX_LEN_SEGMENT = 256
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
MARGIN = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset
class ContrastiveDataset(Dataset):
    def __init__(self, json_path):
        with open(json_path, 'r') as f:
            self.data = json.load(f)
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        query = item["query"]
        pos = item["positive_segment"]
        neg = item["negative_segment"]

        q_enc = self.tokenizer(query, truncation=True, padding='max_length', max_length=MAX_LEN_QUERY, return_tensors="pt")
        p_enc = self.tokenizer(pos, truncation=True, padding='max_length', max_length=MAX_LEN_SEGMENT, return_tensors="pt")
        n_enc = self.tokenizer(neg, truncation=True, padding='max_length', max_length=MAX_LEN_SEGMENT, return_tensors="pt")

        return {
            "query_input_ids": q_enc["input_ids"].squeeze(0),
            "query_attention_mask": q_enc["attention_mask"].squeeze(0),
            "pos_input_ids": p_enc["input_ids"].squeeze(0),
            "pos_attention_mask": p_enc["attention_mask"].squeeze(0),
            "neg_input_ids": n_enc["input_ids"].squeeze(0),
            "neg_attention_mask": n_enc["attention_mask"].squeeze(0)
        }

# Model
class DualEncoderModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0, :]  # [CLS] token

# Loss
def contrastive_loss(query, pos, neg, margin):
    sim_pos = nn.functional.cosine_similarity(query, pos)
    sim_neg = nn.functional.cosine_similarity(query, neg)
    loss = torch.clamp(margin + sim_neg - sim_pos, min=0.0)
    return loss.mean(), (sim_pos > sim_neg).float().mean()

# Training loop
def train(json_path):
    dataset = ContrastiveDataset(json_path)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = DualEncoderModel().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(dataloader) * EPOCHS)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        total_acc = 0
        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
            optimizer.zero_grad()
            q = model(batch["query_input_ids"].to(DEVICE), batch["query_attention_mask"].to(DEVICE))
            p = model(batch["pos_input_ids"].to(DEVICE), batch["pos_attention_mask"].to(DEVICE))
            n = model(batch["neg_input_ids"].to(DEVICE), batch["neg_attention_mask"].to(DEVICE))

            loss, acc = contrastive_loss(q, p, n, margin=MARGIN)
            loss.backward()
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            total_acc += acc.item()

        avg_loss = total_loss / len(dataloader)
        avg_acc = total_acc / len(dataloader)
        print(f"✅ Epoch {epoch+1} - Loss: {avg_loss:.4f}, Pairwise Accuracy: {avg_acc:.4f}")
    return model
model = []
if __name__ == "__main__":
    model = train("/content/drive/MyDrive/QBText/train_pairs.json")  # Dosya adını ihtiyacına göre değiştir


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

✅ Epoch 1 - Loss: 0.4046, Pairwise Accuracy: 0.6436


Epoch 2: 100%|██████████| 858/858 [02:50<00:00,  5.02it/s]


✅ Epoch 2 - Loss: 0.2437, Pairwise Accuracy: 0.8145


Epoch 3: 100%|██████████| 858/858 [02:49<00:00,  5.05it/s]

✅ Epoch 3 - Loss: 0.1552, Pairwise Accuracy: 0.8979


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
import json
import random
from tqdm import tqdm

# Config
MODEL_NAME = "bert-base-uncased"
MAX_LEN_QUERY = 32
MAX_LEN_SEGMENT = 256
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
MARGIN = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset
class ContrastiveDataset(Dataset):
    def __init__(self, json_path):
        with open(json_path, 'r') as f:
            self.data = json.load(f)
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        query = item["query"]
        pos = item["positive_segment"]
        neg = item["negative_segment"]

        q_enc = self.tokenizer(query, truncation=True, padding='max_length', max_length=MAX_LEN_QUERY, return_tensors="pt")
        p_enc = self.tokenizer(pos, truncation=True, padding='max_length', max_length=MAX_LEN_SEGMENT, return_tensors="pt")
        n_enc = self.tokenizer(neg, truncation=True, padding='max_length', max_length=MAX_LEN_SEGMENT, return_tensors="pt")

        return {
            "query_input_ids": q_enc["input_ids"].squeeze(0),
            "query_attention_mask": q_enc["attention_mask"].squeeze(0),
            "pos_input_ids": p_enc["input_ids"].squeeze(0),
            "pos_attention_mask": p_enc["attention_mask"].squeeze(0),
            "neg_input_ids": n_enc["input_ids"].squeeze(0),
            "neg_attention_mask": n_enc["attention_mask"].squeeze(0)
        }

# Model
class DualEncoderModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0, :]  # [CLS] token

# Loss
def contrastive_loss(query, pos, neg, margin):
    sim_pos = nn.functional.cosine_similarity(query, pos)
    sim_neg = nn.functional.cosine_similarity(query, neg)
    loss = torch.clamp(margin + sim_neg - sim_pos, min=0.0)
    return loss.mean(), (sim_pos > sim_neg).float().mean()

# Training loop
def train(train_json_path, val_json_path):
    train_dataset = ContrastiveDataset(train_json_path)
    val_dataset = ContrastiveDataset(val_json_path)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = DualEncoderModel().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0,
                                                num_training_steps=len(train_loader) * EPOCHS)

    for epoch in range(EPOCHS):
        # ---- TRAIN ----
        model.train()
        train_loss = 0
        train_acc = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
            optimizer.zero_grad()
            q = model(batch["query_input_ids"].to(DEVICE), batch["query_attention_mask"].to(DEVICE))
            p = model(batch["pos_input_ids"].to(DEVICE), batch["pos_attention_mask"].to(DEVICE))
            n = model(batch["neg_input_ids"].to(DEVICE), batch["neg_attention_mask"].to(DEVICE))

            loss, acc = contrastive_loss(q, p, n, margin=MARGIN)
            loss.backward()
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()
            train_acc += acc.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_train_acc = train_acc / len(train_loader)

        # ---- VALIDATION ----
        model.eval()
        val_loss = 0
        val_acc = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
                q = model(batch["query_input_ids"].to(DEVICE), batch["query_attention_mask"].to(DEVICE))
                p = model(batch["pos_input_ids"].to(DEVICE), batch["pos_attention_mask"].to(DEVICE))
                n = model(batch["neg_input_ids"].to(DEVICE), batch["neg_attention_mask"].to(DEVICE))

                loss, acc = contrastive_loss(q, p, n, margin=MARGIN)
                val_loss += loss.item()
                val_acc += acc.item()

        avg_val_loss = val_loss / len(val_loader)
        avg_val_acc = val_acc / len(val_loader)

        print(f"✅ Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.4f} | "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.4f}")

    return model
model = []
if __name__ == "__main__":
    model = train("/content/drive/MyDrive/QBText/train_pairs.json","/content/drive/MyDrive/QBText/val_pairs.json")  # Dosya adını ihtiyacına göre değiştir


Epoch 1 [Val]: 100%|██████████| 196/196 [00:15<00:00, 12.69it/s]


✅ Epoch 1 | Train Loss: 0.3999, Train Acc: 0.6591 | Val Loss: 0.3802, Val Acc: 0.6569


Epoch 2 [Val]: 100%|██████████| 196/196 [00:15<00:00, 12.39it/s]


✅ Epoch 2 | Train Loss: 0.2419, Train Acc: 0.8096 | Val Loss: 0.3756, Val Acc: 0.6709


Epoch 3 [Val]: 100%|██████████| 196/196 [00:15<00:00, 12.75it/s]

✅ Epoch 3 | Train Loss: 0.1584, Train Acc: 0.8909 | Val Loss: 0.3780, Val Acc: 0.6716


In [ ]:
# Eğitim bittiğinde modeli kaydet
save_path = "/content/drive/MyDrive/QBText/dual_encoder_model.pt"
torch.save(model.state_dict(), save_path)
print(f"📦 Model başarıyla kaydedildi: {save_path}")


📦 Model başarıyla kaydedildi: /content/drive/MyDrive/QBText/dual_encoder_model.pt


INFERENCE

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import json

# Ayarlar
MODEL_NAME = "bert-base-uncased"
MAX_LEN_QUERY = 32
MAX_LEN_SEGMENT = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "/content/drive/MyDrive/QBText/dual_encoder_model.pt"


In [ ]:
class DualEncoderModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0, :]  # [CLS] token


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = DualEncoderModel().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()


DualEncoderModel(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, eleme

In [ ]:
sims = []
def get_best_segment(query, segments):
    # Query encode
    query_inputs = tokenizer(query, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_LEN_QUERY).to(DEVICE)
    with torch.no_grad():
        query_emb = model(query_inputs["input_ids"], query_inputs["attention_mask"])

    # Her segment için benzerlik hesapla
    best_score = -1e9
    best_segment = None

    for seg in segments:
        seg_inputs = tokenizer(seg, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_LEN_SEGMENT).to(DEVICE)
        with torch.no_grad():
            seg_emb = model(seg_inputs["input_ids"], seg_inputs["attention_mask"])

        sim = F.cosine_similarity(query_emb, seg_emb).item()
        sims.append(sim)
        if sim > best_score:
            best_score = sim
            best_segment = seg

    return best_segment, best_score,sims


In [ ]:
query = "dancing"
candidate_segments = [
    "The group started clapping in rhythm.",
    "She explained the data analysis.",
    "They danced to loud music on stage.",
    "A man was reading the newspaper.",
    "He entered the kitchen to cook dinner."
]

best_seg, score,sims = get_best_segment(query, candidate_segments)
print(sims)
print(f"🔍 Query: {query}")
print(f"✅ Best Segment: {best_seg}")
print(f"📊 Similarity Score: {score:.4f}")


[0.3794059455394745, 0.0502726286649704, 0.6817289590835571, 0.2234700322151184, 0.015318498015403748]
🔍 Query: dancing
✅ Best Segment: They danced to loud music on stage.
📊 Similarity Score: 0.6817


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import precision_recall_fscore_support
import json
import numpy as np
from tqdm import tqdm

# Ayarlar
MODEL_PATH = "/content/drive/MyDrive/QBText/dual_encoder_model.pt"
TEST_JSON_PATH = "/content/drive/MyDrive/QBText/test_pairs.json"
MODEL_NAME = "bert-base-uncased"
MAX_LEN_QUERY = 32
MAX_LEN_SEGMENT = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model tanımı
class DualEncoderModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0, :]  # CLS

# Tokenizer ve model yükle
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DualEncoderModel().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# Değerlendirme fonksiyonu
def evaluate_model(test_data, thresholds=[0.25, 0.3, 0.35, 0.4, 0.45, 0.5]):
    results = {}
    for threshold in thresholds:
        y_true = []
        y_pred = []

        for item in tqdm(test_data, desc=f"Threshold {threshold:.2f}"):
            query = item["query"]
            pos_seg = item["positive_segment"]
            neg_seg = item["negative_segment"]

            with torch.no_grad():
                # Encode query
                q_enc = tokenizer(query, truncation=True, padding='max_length',
                                  max_length=MAX_LEN_QUERY, return_tensors="pt").to(DEVICE)
                q_vec = model(q_enc["input_ids"], q_enc["attention_mask"])

                # Encode positive
                p_enc = tokenizer(pos_seg, truncation=True, padding='max_length',
                                  max_length=MAX_LEN_SEGMENT, return_tensors="pt").to(DEVICE)
                p_vec = model(p_enc["input_ids"], p_enc["attention_mask"])

                # Encode negative
                n_enc = tokenizer(neg_seg, truncation=True, padding='max_length',
                                  max_length=MAX_LEN_SEGMENT, return_tensors="pt").to(DEVICE)
                n_vec = model(n_enc["input_ids"], n_enc["attention_mask"])

                # Cosine similarity
                sim_pos = nn.functional.cosine_similarity(q_vec, p_vec).item()
                sim_neg = nn.functional.cosine_similarity(q_vec, n_vec).item()

                # Prediction
                pred_pos = int(sim_pos >= threshold)
                pred_neg = int(sim_neg >= threshold)

                y_true.extend([1, 0])
                y_pred.extend([pred_pos, pred_neg])

        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
        results[threshold] = {
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

    return results

# Test verisini yükle
with open(TEST_JSON_PATH, "r") as f:
    test_data = json.load(f)

# Değerlendir
scores = evaluate_model(test_data)

# Sonuçları yazdır
for th, metrics in scores.items():
    print(f"Threshold {th:.2f} | Precision: {metrics['precision']:.3f} | Recall: {metrics['recall']:.3f} | F1: {metrics['f1']:.3f}")


Threshold 0.50: 100%|██████████| 1428/1428 [00:39<00:00, 36.22it/s]

Threshold 0.25 | Precision: 0.655 | Recall: 0.496 | F1: 0.565
Threshold 0.30 | Precision: 0.666 | Recall: 0.456 | F1: 0.541
Threshold 0.35 | Precision: 0.684 | Recall: 0.410 | F1: 0.513
Threshold 0.40 | Precision: 0.696 | Recall: 0.363 | F1: 0.477
Threshold 0.45 | Precision: 0.710 | Recall: 0.322 | F1: 0.443
Threshold 0.50 | Precision: 0.716 | Recall: 0.270 | F1: 0.392


TEST

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import precision_recall_fscore_support
import json
from tqdm import tqdm

# Config
MODEL_PATH = "/content/drive/MyDrive/QBText/dual_encoder_model.pt"
TEST_JSON_PATH = "/content/drive/MyDrive/QBText/test_pairs.json"
MODEL_NAME = "bert-base-uncased"
MAX_LEN_QUERY = 32
MAX_LEN_SEGMENT = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
THRESHOLDS = [0.25, 0.5, 0.6, 0.75]

# Model Definition
class DualEncoderModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0, :]

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DualEncoderModel().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# Load test data
with open(TEST_JSON_PATH, "r") as f:
    test_data = json.load(f)

# Evaluation
def evaluate_thresholds(test_data, thresholds):
    results = {}

    for threshold in thresholds:
        y_true = []
        y_pred = []

        for item in tqdm(test_data, desc=f"Threshold: {threshold}"):
            query = item["query"]
            pos = item["positive_segment"]
            neg = item["negative_segment"]

            # Encode
            q_enc = tokenizer(query, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_LEN_QUERY).to(DEVICE)
            pos_enc = tokenizer(pos, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_LEN_SEGMENT).to(DEVICE)
            neg_enc = tokenizer(neg, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_LEN_SEGMENT).to(DEVICE)

            with torch.no_grad():
                q_vec = model(q_enc["input_ids"], q_enc["attention_mask"])
                pos_vec = model(pos_enc["input_ids"], pos_enc["attention_mask"])
                neg_vec = model(neg_enc["input_ids"], neg_enc["attention_mask"])

                sim_pos = nn.functional.cosine_similarity(q_vec, pos_vec).item()
                sim_neg = nn.functional.cosine_similarity(q_vec, neg_vec).item()

            # Ground truth: 1 for pos, 0 for neg
            y_true.extend([1, 0])
            y_pred.extend([
                1 if sim_pos >= threshold else 0,
                1 if sim_neg >= threshold else 0
            ])

        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
        results[threshold] = {
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

    return results

# Run Evaluation
metrics = evaluate_thresholds(test_data, THRESHOLDS)
print("\n📊 F1 Results by Threshold:")
for t, scores in metrics.items():
    print(f"Threshold {t:.2f} -> Precision: {scores['precision']:.3f}, Recall: {scores['recall']:.3f}, F1: {scores['f1']:.3f}")


Threshold: 0.75: 100%|██████████| 1428/1428 [00:39<00:00, 36.00it/s]


📊 F1 Results by Threshold:
Threshold 0.25 -> Precision: 0.652, Recall: 0.543, F1: 0.593
Threshold 0.50 -> Precision: 0.714, Recall: 0.308, F1: 0.431
Threshold 0.60 -> Precision: 0.748, Recall: 0.224, F1: 0.345
Threshold 0.75 -> Precision: 0.815, Recall: 0.099, F1: 0.176


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import average_precision_score
import json
from tqdm import tqdm
import numpy as np

# Config
MODEL_PATH = "/content/drive/MyDrive/QBText/dual_encoder_model.pt"
TEST_PATH = "/content/drive/MyDrive/QBText/test_pairs.json"
MODEL_NAME = "bert-base-uncased"
MAX_LEN_QUERY = 32
MAX_LEN_SEGMENT = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model tanımı (Eğitimle aynı)
class DualEncoderModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

    def forward(self, input_ids, attention_mask):
        output = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return output.last_hidden_state[:, 0, :]  # [CLS] token

# Tokenizer ve model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DualEncoderModel().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# Veriyi oku
with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

# Her query için AP hesapla
query_map_scores = {}

# Query başına grupla
query_groups = {}
for item in test_data:
    query = item["query"]
    if query not in query_groups:
        query_groups[query] = []
    query_groups[query].append(item)

for query, group in tqdm(query_groups.items(), desc="Evaluating queries"):
    q_enc = tokenizer(query, padding="max_length", truncation=True, max_length=MAX_LEN_QUERY, return_tensors="pt")
    q_input_ids = q_enc["input_ids"].to(DEVICE)
    q_attn_mask = q_enc["attention_mask"].to(DEVICE)
    with torch.no_grad():
        q_vec = model(q_input_ids, q_attn_mask)  # [1, hidden_size]

    similarities = []
    labels = []

    for item in group:
        for label, seg_key in [(1, "positive_segment"), (0, "negative_segment")]:
            s_enc = tokenizer(item[seg_key], padding="max_length", truncation=True, max_length=MAX_LEN_SEGMENT, return_tensors="pt")
            s_input_ids = s_enc["input_ids"].to(DEVICE)
            s_attn_mask = s_enc["attention_mask"].to(DEVICE)
            with torch.no_grad():
                s_vec = model(s_input_ids, s_attn_mask)  # [1, hidden_size]
            sim = nn.functional.cosine_similarity(q_vec, s_vec).item()
            similarities.append(sim)
            labels.append(label)

    if len(set(labels)) > 1:  # En az bir pozitif bir negatif varsa
        ap = average_precision_score(labels, similarities)
        query_map_scores[query] = ap

# Sonuç
all_aps = list(query_map_scores.values())
mean_ap = np.mean(all_aps)
print(f"\n✅ Mean Average Precision (mAP): {mean_ap:.4f}")


Evaluating queries: 100%|██████████| 139/139 [00:29<00:00,  4.64it/s]


✅ Mean Average Precision (mAP): 0.7291


In [ ]:
!pip install rouge-score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=7f62e814c18329b904ee9a59543c240bd8ae6d282bdd789b28eedac02e3d73e5
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from rouge_score import rouge_scorer
import json
from tqdm import tqdm

# Ayarlar
MODEL_PATH = "/content/drive/MyDrive/QBText/dual_encoder_model.pt"
TEST_JSON = "/content/drive/MyDrive/QBText/test_pairs.json"
MODEL_NAME = "bert-base-uncased"
MAX_LEN_QUERY = 32
MAX_LEN_SEGMENT = 256
SIM_THRESHOLD = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model
class DualEncoderModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0, :]  # [CLS]

# Tokenizer ve model yükle
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DualEncoderModel().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores_list = []

# Test verisini oku
with open(TEST_JSON, "r") as f:
    test_data = json.load(f)

# Query'lere göre grupla
grouped = {}
for item in test_data:
    query = item["query"]
    if query not in grouped:
        grouped[query] = []
    grouped[query].append(item)

# Her query için özet tahmini ve ROUGE hesapla
for query, group in tqdm(grouped.items(), desc="Evaluating ROUGE"):
    q_enc = tokenizer(query, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_LEN_QUERY).to(DEVICE)
    with torch.no_grad():
        q_vec = model(q_enc["input_ids"], q_enc["attention_mask"])

    ref_summary = []
    pred_summary = []

    for item in group:
        for label, key in [(1, "positive_segment"), (0, "negative_segment")]:
            text = item[key]
            seg_enc = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_LEN_SEGMENT).to(DEVICE)
            with torch.no_grad():
                seg_vec = model(seg_enc["input_ids"], seg_enc["attention_mask"])
            sim = nn.functional.cosine_similarity(q_vec, seg_vec).item()
            if label == 1:
                ref_summary.append(text)
            if sim >= SIM_THRESHOLD:
                pred_summary.append(text)

    if ref_summary and pred_summary:
        reference = " ".join(ref_summary)
        prediction = " ".join(pred_summary)
        score = scorer.score(reference, prediction)
        scores_list.append(score)

# Ortalama skorları hesapla
def average_score(metric):
    return sum([s[metric].fmeasure for s in scores_list]) / len(scores_list)

print("\n📊 ROUGE Results (Threshold = 0.5)")
print(f"ROUGE-1: {average_score('rouge1'):.4f}")
print(f"ROUGE-2: {average_score('rouge2'):.4f}")
print(f"ROUGE-L: {average_score('rougeL'):.4f}")


Evaluating ROUGE: 100%|██████████| 139/139 [01:38<00:00,  1.41it/s]


📊 ROUGE Results (Threshold = 0.5)
ROUGE-1: 0.5804
ROUGE-2: 0.5073
ROUGE-L: 0.5117


In [ ]:
!pip install pytube

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.1 MB/s eta 0:00:00
